# TAN and SPY daily prices (2016–2024)

Downloads unadjusted daily OHLC and adjusted close from Yahoo Finance via `yfinance` for:

- **TAN** — Invesco Solar ETF
- **SPY** — market control

The requested window is January 1, 2016 through December 31, 2024 (inclusive).

In [ ]:
# Uncomment if yfinance is not installed in this kernel:
# %pip install -q yfinance

from pathlib import Path

import pandas as pd
import yfinance as yf

TICKERS = ["TAN", "SPY"]
START = "2016-01-01"
END_EXCLUSIVE = "2025-01-01"  # yfinance's end date is exclusive
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
raw = yf.download(
    TICKERS,
    start=START,
    end=END_EXCLUSIVE,
    interval="1d",
    auto_adjust=False,
    actions=False,
    group_by="ticker",
    progress=False,
    threads=True,
)

if raw.empty:
    raise RuntimeError("Yahoo Finance returned no data.")

raw.head()

In [ ]:
required_columns = ["Open", "High", "Low", "Close", "Adj Close"]
frames = []

for ticker in TICKERS:
    ticker_data = raw[ticker][required_columns].copy()
    ticker_data.columns.name = None
    ticker_data = ticker_data.dropna(how="all").reset_index()
    ticker_data.insert(1, "Ticker", ticker)
    frames.append(ticker_data)

prices = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

prices

In [ ]:
# Validate coverage, uniqueness, and OHLC completeness.
coverage = prices.groupby("Ticker").agg(
    first_date=("Date", "min"),
    last_date=("Date", "max"),
    trading_days=("Date", "size"),
    missing_values=("Open", lambda s: prices.loc[s.index, required_columns].isna().sum().sum()),
)

assert not prices.duplicated(["Ticker", "Date"]).any(), "Duplicate ticker-date rows found."
assert (prices[required_columns].notna().all(axis=1)).all(), "Missing requested price values found."
assert prices["Date"].min() >= pd.Timestamp(START)
assert prices["Date"].max() <= pd.Timestamp("2024-12-31")

coverage

In [ ]:
# Save one tidy combined file and one file per ETF.
combined_path = OUTPUT_DIR / "tan_spy_daily_2016_2024.csv"
prices.to_csv(combined_path, index=False)

for ticker, ticker_data in prices.groupby("Ticker", sort=False):
    ticker_data.drop(columns="Ticker").to_csv(
        OUTPUT_DIR / f"{ticker.lower()}_daily_2016_2024.csv",
        index=False,
    )

print(f"Saved {len(prices):,} rows to {combined_path.resolve()}")
print("Columns:", prices.columns.tolist())

## Polymarket: Trump 2024 election market

Pulls the one-minute-fidelity price history for the **Yes** token in “Will Donald Trump win the 2024 US Presidential Election?” The Gamma API is used to discover the token ID, then the public CLOB `/prices-history` endpoint is queried in 14-day chunks because long high-resolution requests can fail for resolved markets.

In [ ]:
import json
import time

import requests

EVENT_SLUG = "presidential-election-winner-2024"
MARKET_SLUG = "will-donald-trump-win-the-2024-us-presidential-election"
GAMMA_URL = "https://gamma-api.polymarket.com/events"
CLOB_HISTORY_URL = "https://clob.polymarket.com/prices-history"

session = requests.Session()
session.headers.update({"User-Agent": "prediction-market-research/1.0"})

In [ ]:
# Discover the resolved market and map its Yes outcome to the CLOB token ID.
response = session.get(GAMMA_URL, params={"slug": EVENT_SLUG}, timeout=30)
response.raise_for_status()
events = response.json()

matches = [
    market
    for event in events
    for market in event.get("markets", [])
    if market.get("slug") == MARKET_SLUG
]
if len(matches) != 1:
    raise RuntimeError(f"Expected one matching market, found {len(matches)}.")

trump_market = matches[0]
outcomes = json.loads(trump_market["outcomes"])
token_ids = json.loads(trump_market["clobTokenIds"])
yes_token_id = dict(zip(outcomes, token_ids))["Yes"]

market_start = pd.Timestamp(trump_market["startDate"]).tz_convert("UTC")
market_end = pd.Timestamp(trump_market["closedTime"]).tz_convert("UTC")

{
    "question": trump_market["question"],
    "condition_id": trump_market["conditionId"],
    "yes_token_id": yes_token_id,
    "market_start": market_start,
    "market_end": market_end,
}

In [ ]:
def fetch_history_chunk(start, end, max_attempts=4):
    params = {
        "market": yes_token_id,
        "startTs": int(start.timestamp()),
        "endTs": int(end.timestamp()),
        "fidelity": 1,  # minutes
    }
    for attempt in range(max_attempts):
        try:
            response = session.get(CLOB_HISTORY_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json().get("history", [])
        except requests.RequestException:
            if attempt == max_attempts - 1:
                raise
            time.sleep(2**attempt)


# Explicit date ranges work more reliably than interval="max" for resolved markets.
chunk_size = pd.Timedelta(days=14)
chunk_start = market_start.floor("s")
history = []

while chunk_start < market_end:
    chunk_end = min(chunk_start + chunk_size, market_end)
    chunk = fetch_history_chunk(chunk_start, chunk_end)
    history.extend(chunk)
    print(f"{chunk_start.date()} to {chunk_end.date()}: {len(chunk):,} points")
    chunk_start = chunk_end
    time.sleep(0.1)

if not history:
    raise RuntimeError("The Polymarket CLOB API returned no price history.")

In [ ]:
trump_prices_1min = (
    pd.DataFrame(history)
    .rename(columns={"t": "timestamp_unix", "p": "price"})
    .drop_duplicates("timestamp_unix", keep="last")
    .sort_values("timestamp_unix")
    .reset_index(drop=True)
)
trump_prices_1min.insert(
    0,
    "timestamp_utc",
    pd.to_datetime(trump_prices_1min["timestamp_unix"], unit="s", utc=True),
)
trump_prices_1min.insert(2, "outcome", "Yes")
trump_prices_1min.insert(3, "token_id", yes_token_id)

assert trump_prices_1min["timestamp_utc"].is_monotonic_increasing
assert trump_prices_1min["timestamp_unix"].is_unique
assert trump_prices_1min["price"].between(0, 1).all()

polymarket_path = OUTPUT_DIR / "polymarket_trump_2024_yes_1min.csv"
trump_prices_1min.to_csv(polymarket_path, index=False)

print(f"Saved {len(trump_prices_1min):,} points to {polymarket_path.resolve()}")
print(
    f"Coverage: {trump_prices_1min['timestamp_utc'].min()} to "
    f"{trump_prices_1min['timestamp_utc'].max()}"
)
trump_prices_1min

## Final daily merged dataset

ETF returns are simple returns from adjusted close. `p` is the last Polymarket Yes price observed in each UTC day. The merge retains ETF trading dates covered by the prediction market; `Δp` is then calculated across those retained dates, so a Monday change includes any weekend movement since Friday.

In [ ]:
adjusted_close = (
    prices.pivot(index="Date", columns="Ticker", values="Adj Close")
    .sort_index()
)

etf_returns = adjusted_close.pct_change(fill_method=None).rename(
    columns={ticker: f"{ticker} ret" for ticker in TICKERS}
)

polymarket_daily = (
    trump_prices_1min.assign(
        Date=trump_prices_1min["timestamp_utc"].dt.floor("D").dt.tz_localize(None)
    )
    .groupby("Date")["price"]
    .last()
    .rename("p")
)

merged_daily = etf_returns.join(polymarket_daily, how="inner").sort_index()
merged_daily["Δp"] = merged_daily["p"].diff()
merged_daily = merged_daily[["TAN ret", "SPY ret", "p", "Δp"]].dropna()

assert merged_daily.index.is_monotonic_increasing
assert merged_daily.index.is_unique
assert merged_daily.notna().all().all()

merged_path = OUTPUT_DIR / "daily_merged_tan_spy_polymarket_2024.csv"
merged_daily.to_csv(merged_path, index_label="Date")

print(f"Saved {len(merged_daily):,} daily rows to {merged_path.resolve()}")
print(f"Coverage: {merged_daily.index.min().date()} to {merged_daily.index.max().date()}")
merged_daily